# ICA Lens workflow: collect → fit → profile → apply

This notebook runs the complete workflow on **GPT-2 Small, layer 8**, using **100,000 sampled token activations from Pile-10k** and **50 fixed FastICA iterations**. A T4 is sufficient, although capture and profiling take several minutes.

In [ ]:
%pip install -q icalens==0.4.0

In [ ]:
import os, time
from pathlib import Path

assert os.system("nvidia-smi >/dev/null 2>&1") == 0, "Select a GPU runtime and run again."
root = Path("/content/icalens-workflow")
capture = root / "activations"
lens_path = root / "lens"
root.mkdir(exist_ok=True)

def run(label, command):
    print(f"\n{'='*18} {label} {'='*18}")
    started = time.perf_counter()
    status = os.system(command)
    elapsed = time.perf_counter() - started
    if status != 0: raise RuntimeError(f"{label} failed with status {status}")
    print(f"{label} completed in {elapsed/60:.2f} min")
    return elapsed

In [ ]:
collect_seconds = run("COLLECT", f'''icalens capture text \
  --model openai-community/gpt2 \
  --dataset NeelNanda/pile-10k --split train --text-field text \
  --layers 8 --candidate-tokens 100000 --token-budget 100000 \
  --capture-layers-at-once 1 --output {capture}''')
print("Activation artifact:", capture)

In [ ]:
fit_seconds = run("FIT", f'''icalens fit activations \
  --input {capture} --layers 8 --icalens-preprocessing none \
  --max-iter 50 --fit-batch-size 8192 --output {lens_path}''')
print("Fitted Lens:", lens_path)

In [ ]:
profile_seconds = run("PROFILE", f'''icalens profile \
  --lens {lens_path} --layers 8 \
  --dataset NeelNanda/pile-10k --split train --text-field text \
  --max-tokens 100000 --top-k-examples 20''')
print("Profiled Lens:", lens_path)

In [ ]:
from icalens import ICALens

started = time.perf_counter()
lens = ICALens.from_pretrained(lens_path)
profiles = [lens.component_profile(layer=8, component=c) for c in range(768)]
component = max(range(768), key=lambda c: profiles[c]["score_statistics"]["excess_kurtosis"])
profile = profiles[component]
print(f"Most heavy-tailed component: C{component}")
print("Selected semantic tail:", profile["tail_direction"])
display(profile)

held_out = "The API requires each request to include a valid JSON object and authentication header."
analysis = lens.analyze(held_out, layer=8, selected_components=[component])
display(analysis)
apply_seconds = time.perf_counter() - started
print(f"APPLY completed in {apply_seconds:.1f} s")
print({"collect_min": collect_seconds/60, "fit_min": fit_seconds/60, "profile_min": profile_seconds/60, "apply_s": apply_seconds})